# SystemTopology `remove_node`

Regression test for `SystemTopology::removeNode`
([#529](https://github.com/sogno-platform/dpsim/issues/529)): removing a node
must also remove the components connected to it, keep unrelated components,
and leave the per-node component map (`components_at_node`) consistent.

In [ ]:
import dpsimpy


def comp_names(system):
    return sorted(c.name() for c in system.components)


def node_names(system):
    return sorted(n.name() for n in system.nodes)

In [ ]:
gnd = dpsimpy.emt.SimNode.gnd
n1 = dpsimpy.emt.SimNode("n1")
n2 = dpsimpy.emt.SimNode("n2")

vs = dpsimpy.emt.ph1.VoltageSource("vs")
vs.set_parameters(V_ref=complex(10, 0), f_src=50)
r1 = dpsimpy.emt.ph1.Resistor("r1")
r1.set_parameters(R=1)
c1 = dpsimpy.emt.ph1.Capacitor("c1")
c1.set_parameters(C=1e-3)

vs.connect([gnd, n1])
r1.connect([n1, n2])
c1.connect([n2, gnd])

system = dpsimpy.SystemTopology(50, [gnd, n1, n2], [vs, r1, c1])
assert comp_names(system) == ["c1", "r1", "vs"]
print("before:", node_names(system), comp_names(system))

In [ ]:
system.remove_node("n2")
print("after:", node_names(system), comp_names(system))

assert "n2" not in node_names(system)
assert comp_names(system) == ["vs"]

for node, comps in system.components_at_node.items():
    assert node.name() != "n2", "stale map entry for removed node"
    for comp in comps:
        assert comp.name() not in ("r1", "c1"), "stale map reference"
print("connected components removed, map consistent: ok")

In [ ]:
system.remove_node("does_not_exist")
assert comp_names(system) == ["vs"]
print("removing a non-existent node is a no-op: ok")

In [ ]:
k1 = dpsimpy.emt.SimNode("k1")
k2 = dpsimpy.emt.SimNode("k2")
k3 = dpsimpy.emt.SimNode("k3")
rx = dpsimpy.emt.ph1.Resistor("rx")
rx.set_parameters(R=1)
rx.connect([k1, k2])
ry = dpsimpy.emt.ph1.Resistor("ry")
ry.set_parameters(R=1)
ry.connect([k2, k3])

s2 = dpsimpy.SystemTopology(50, [k1, k2, k3], [rx, ry])
s2.remove_node("k3")

assert comp_names(s2) == ["rx"]
at_k2 = [
    c.name()
    for node, comps in s2.components_at_node.items()
    if node.name() == "k2"
    for c in comps
]
assert "rx" in at_k2
print("component sharing only another node survives: ok")